In [ ]:
# Previous_Rv1/figure_scripts/Fig1_D_histogram_S1_E.ipynb, run by ../FigS2_AF_robustness.ipynb (final_figs.run_rv1_capture) on
# non-overlapping windows (per session, all states in time order, a window kept only if its centre is at least
# one window length after the last kept one). Thinned or rebuilt tables: CSV; the others: RV1_CSV.
import os, glob, pickle, re, sys   # standard imports
from pathlib import Path
sys.path.insert(0, str(Path('..') / '..' / 'analysis_scripts' / 'src'))   # the criticality library
import pandas as pd
import numpy as np
CSV = Path('..') / '..' / 'csv' / 'FigS2_AF_robustness' / 'rv1_Fig1_D_histogram_S1_E'    # thinned / rebuilt tables
RV1_CSV = Path('..') / '..' / '..' / 'Previous_Rv1' / 'csv' / 'Fig1_D_histogram_S1_E'   # the others
OUT = Path('..') / '..' / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """A table of this notebook (round_trip: the floats exactly as written)."""
    return pd.read_csv(CSV / name if (CSV / name).exists() else RV1_CSV / name, float_precision='round_trip', **kw)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
import os, re, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------ I/O helpers ------------------
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(mat_path, squeeze_me=True, struct_as_record=False, simplify_cells=True)
    except TypeError:
        return loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)

def _to_1d_array(x):
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)
    if arr.ndim == 0:
        return np.atleast_1d(arr).astype(float, copy=False)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def _infer_animal_day_from_path(basepath):
    base = os.path.basename(basepath.rstrip("\\/"))
    animal = os.path.basename(os.path.dirname(basepath))
    mday = re.search(r"day(\d+)", base, flags=re.I)
    day = int(mday.group(1)) if mday else None
    return animal, day

# ------------------ extract center_s + d2 ------------------
def _extract_center_d2_by_state_from_mat(mat_path, region=None, animal=None, day=None):
    """
    Returns DataFrame with columns:
      state, center_s, d2, region, animal, day
    """
    mat = _load_mat_any(mat_path)
    results = mat.get("results", None)
    if results is None:
        return pd.DataFrame(columns=["state","center_s","d2","region","animal","day"])

    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    rows = []
    for it in iter_results:
        if hasattr(it, "get"):
            st = _to_state_str(it.get("state", "UNKNOWN"))
            c_raw = it.get("center", [])
            d2_raw = it.get("d2", [])
        elif isinstance(it, dict) or hasattr(it, "__getitem__"):
            st = _to_state_str(it.get("state", it["state"] if "state" in it else "UNKNOWN"))
            c_raw = it.get("center", it["center"] if "center" in it else [])
            d2_raw = it.get("d2", it["d2"] if "d2" in it else [])
        else:
            st, c_raw, d2_raw = "UNKNOWN", [], []

        centers = _to_1d_array(c_raw)
        d2_vals = _to_1d_array(d2_raw)
        ok = np.isfinite(centers) & np.isfinite(d2_vals)
        centers, d2_vals = centers[ok], d2_vals[ok]
        if centers.size == 0:
            continue

        st_up = _to_state_str(st).upper()
        if st_up == "WAKE":
            st_up = "AWAKE"
        if st_up not in ("NREM", "AWAKE", "REM"):
            continue

        rows.append(pd.DataFrame({
            "state": st_up,
            "center_s": centers.astype(float, copy=False),
            "d2": d2_vals.astype(float, copy=False),
            "region": region if region else "",
            "animal": animal if animal else "",
            "day": day if day is not None else ""
        }))

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=["state","center_s","d2","region","animal","day"]
    )

def load_center_d2_from_session(basepath, region):
    """
    Loads newest 30s and newest 60s files (if any) and returns pooled
    center_s + d2 with a 'window' column ∈ {'30s','60s'}.
    """
    region = region.upper()
    search_dir = Path(basepath) / "complex_system"
    if not search_dir.exists():
        return pd.DataFrame(columns=["state","center_s","d2","window","region","animal","day"])

    pats = {
        "30s": f"*_{region}_D2_time_resolved_error_bar.mat",
    }
    animal, day = _infer_animal_day_from_path(basepath)

    out = []
    for win_label, pat in pats.items():
        files = glob.glob(str(search_dir / pat))
        if not files:
            continue
        mat_path = max(files, key=os.path.getmtime)
        df = _extract_center_d2_by_state_from_mat(mat_path, region=region, animal=animal, day=day)
        if df.empty:
            continue
        df["window"] = win_label
        out.append(df)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(
        columns=["state","center_s","d2","window","region","animal","day"]
    )

# --- compute per-row durations from timestamps ---
def add_duration_from_timestamps(df, fill_last="median"):
    """
    For each (animal, day, window, state), sort by center_s and compute row-wise
    duration as the forward difference: duration[i] = center[i+1] - center[i].
    The last row per group gets:
        - fill_last="median": group's median Δt
        - fill_last="drop": NaN (will be dropped later)
        - fill_last=float: a constant seconds value
    Returns a copy with a new 'duration_s' column.
    """
    d = df.copy()
    grp_cols = ["animal","day","window","state"]

    def _durations(g):
        g = g.sort_values("center_s")
        c = g["center_s"].to_numpy(dtype=float)
        dt = np.empty_like(c, dtype=float)
        if c.size == 1:
            dt[:] = np.nan
        else:
            dt[:-1] = np.diff(c)
            # last-row handling
            if fill_last == "median":
                med = np.nanmedian(dt[:-1]) if dt.size > 1 else np.nan
                dt[-1] = med if np.isfinite(med) and med > 0 else np.nan
            elif fill_last == "drop":
                dt[-1] = np.nan
            else:
                # numeric
                try:
                    dt[-1] = float(fill_last)
                except Exception:
                    dt[-1] = np.nan
        # negative/zero dt are invalid
        dt[~np.isfinite(dt) | (dt <= 0)] = np.nan
        g["duration_s"] = dt
        return g

    d = d.groupby(grp_cols, observed=True, group_keys=False).apply(_durations)
    # drop rows with no valid duration
    d = d[np.isfinite(d["duration_s"]) & (d["duration_s"] > 0)]
    return d

def compute_equal_width_bins(d, n_bins=10, by_window=False):
    """
    Equal-width bins on D² within each state (and optionally state×window).
    Adds 'd2_bin' ∈ {1..n_bins} as nullable Int64.
    Returns df_with_bins, edges_map.
    """
    d = d.copy()

    # Make sure d2 is numeric
    d["d2"] = pd.to_numeric(d["d2"], errors="coerce")

    # Pre-create a nullable integer column to avoid dtype thrash
    d["d2_bin"] = pd.Series([pd.NA] * len(d), dtype="Int64")

    edges_map = {}
    groups = (
        d.groupby(["state", "window"], observed=True)
        if by_window else
        d.groupby(["state"], observed=True)
    )

    for key, g in groups:
        idx = g.index
        vals = d.loc[idx, "d2"].to_numpy()

        if vals.size == 0 or not np.isfinite(vals).any():
            continue

        dmin, dmax = float(np.nanmin(vals)), float(np.nanmax(vals))
        if not np.isfinite(dmin) or not np.isfinite(dmax) or dmin == dmax:
            # degenerate case: single bin
            edges = np.array([dmin - 1e-12, dmax + 1e-12], dtype=float)
        else:
            edges = np.linspace(dmin, dmax, n_bins + 1, dtype=float)

        edges_map[key] = edges

        # labels=False -> integer codes [0..n_bins-1], NaN for out-of-bin
        codes = pd.cut(
            d.loc[idx, "d2"],
            bins=edges,
            labels=False,
            include_lowest=True
        )

        # Convert to nullable Int64 with 1-based labels
        # Note: adding 1 keeps NA as NA with pandas nullable integers
        codes = (pd.Series(codes, index=idx, dtype="Int64") + 1).astype("Int64")

        # Single assignment of a clean integer Series
        d.loc[idx, "d2_bin"] = codes

    # Ensure final dtype
    d["d2_bin"] = d["d2_bin"].astype("Int64")
    return d, edges_map

def aggregate_durations_by_bin(
    d,
    by_window=False,
    include_env=False,
    normalize_within="state",   # "state" or None
):
    """
    Sum durations per d2_bin, grouped by:
        - state
        - d2_bin
        - [environment] (optional)
        - [window]      (optional)

    If include_env=False, `environment` is ignored and not in the output.

    normalize_within:
        - "state": normalize within each (state, [window], [environment if included])
        - None: keep absolute seconds
    """
    d = d.copy()

    # ---- GROUPING KEYS ----
    keys = ["state", "d2_bin"]
    if include_env and ("environment" in d.columns):
        keys.append("environment")
    if by_window and ("window" in d.columns):
        keys.append("window")

    # ---- AGGREGATE ----
    agg = (
        d.dropna(subset=["d2_bin", "duration_s"])
         .groupby(keys, observed=True)["duration_s"]
         .sum()
         .rename("dwell_sec_total")
         .reset_index()
    )

    # ---- NORMALIZATION ----
    if normalize_within == "state":
        norm_keys = ["state"]
        if by_window and ("window" in agg.columns):
            norm_keys.append("window")
        if include_env and ("environment" in agg.columns):
            norm_keys.append("environment")

        agg["dwell_frac"] = agg.groupby(norm_keys)["dwell_sec_total"].transform(
            lambda x: x / x.sum() if x.sum() > 0 else x
        )

    return agg


def plot_histograms(agg, by_window=False, use_fraction=True, n_bins=10, title=None, save_path=None):
    """
    Bar histograms of dwell (fraction or seconds) vs D² bin for each state.
    If by_window=True, facet by 30s/60s columns.
    """
    order_states = ["NREM","AWAKE","REM"]
    ycol = "dwell_frac" if use_fraction else "dwell_sec_total"
    ylab = "Dwell fraction" if use_fraction else "Total dwell (s)"

    if by_window and "window" in agg.columns:
        order_windows = ["30s"]
        windows = [w for w in order_windows if w in agg["window"].unique()]
        fig, axes = plt.subplots(len(order_states), len(windows), figsize=(5*len(windows), 3.8*len(order_states)),
                                 sharex=True, sharey=True)
        if len(order_states) == 1: axes = np.atleast_2d(axes)
        for i, st in enumerate(order_states):
            for j, w in enumerate(windows):
                ax = axes[i, j]
                sub = agg[(agg["state"]==st) & (agg["window"]==w) & agg["d2_bin"].notna()]
                if sub.empty:
                    ax.set_title(f"{st} – {w} (no data)")
                    continue
                ax.bar(sub["d2_bin"].astype(int), sub[ycol], width=0.7, edgecolor="black")
                if i == 0: ax.set_title(w, fontweight="bold")
                if j == 0: ax.set_ylabel(f"{st}\n{ylab}")
                ax.set_xlabel("D² bin")
                ax.set_xticks(range(1, n_bins+1))
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
        plt.suptitle(title or "Dwell vs D² bins (by window)", fontweight="bold")
    else:
        fig, axes = plt.subplots(1, len(order_states), figsize=(5*len(order_states), 4), sharey=True)
        if len(order_states) == 1: axes = [axes]
        for ax, st in zip(axes, order_states):
            sub = agg[(agg["state"]==st) & agg["d2_bin"].notna()]
            if sub.empty:
                ax.set_title(f"{st} (no data)")
                continue
            ax.bar(sub["d2_bin"].astype(int), sub[ycol], width=0.7, edgecolor="black")
            ax.set_title(st, fontweight="bold")
            ax.set_xlabel("D² bin")
            ax.set_xticks(range(1, n_bins+1))
            if ax is axes[0]: ax.set_ylabel(ylab)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
        plt.suptitle(title or "Dwell vs D² bins (pooled)", fontweight="bold")

    plt.tight_layout(rect=[0,0,1,0.95])
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300)
        print(f"Saved: {save_path}")
    plt.show()


# ------------------ DRIVER ------------------
# Your session folders:
paths = [
    r"W:\data\PPP\PPP20\day4",
    r"W:\data\PPP\PPP20\day5",
    r"W:\data\PPP\PPP20\day7",
    r"W:\data\PPP\PPP20\day8",
    r"W:\data\PPP\PPP20\day9",
    r"W:\data\PPP\PPP21\day4",
    r"W:\data\PPP\PPP21\day5",
    r"W:\data\PPP\PPP21\day7",
    r"W:\data\PPP\PPP21\day8",
    r"W:\data\PPP\PPP8\day8",
    r"W:\data\PPP\PPP8\day14",
    r"W:\data\PPP\PPP4\day11",
    r"W:\data\PPP\PPP7\day7",
    r"W:\data\PPP\PPP7\day10",
    r"W:\data\PPP\PPP7\day12",
    r"W:\data\PPP\PPP7\day14",
    r"W:\data\PPP\PPP23\day7",
    r"W:\data\PPP\PPP23\day8",
]
regions = ["CA1"]  # change if needed
save_root = r"../figure_output"

def _find_state_mat(basepath):
    """
    Heuristics to locate a MAT file holding state_dict/THETA.
    Searches common names in basepath and subfolders.
    """
    candidates = []
    basepath = Path(basepath)
    # common names / locations; extend as needed
    pats = [
        "*state*.mat",
        "*State*.mat",
        "states.mat",
        "state_dict.mat",
        "sleep_states.mat",
        str(basepath / "complex_system" / "*state*.mat"),
        str(basepath / "complex_system" / "state_dict.mat"),
    ]
    for pat in pats:
        candidates.extend(glob.glob(str(basepath / pat) if not pat.startswith(str(basepath)) else pat))
    # fall back: any .mat in basepath
    if not candidates:
        candidates.extend(glob.glob(str(basepath / "*.mat")))
    # pick the newest that actually contains THETA
    best = None
    best_mtime = -1
    for p in candidates:
        try:
            m = _load_mat_any(p)
            # look for THETA under state_dict or top-level
            has_theta = False
            if "state_dict" in m and isinstance(m["state_dict"], dict):
                has_theta = "THETA" in m["state_dict"]
            if ("THETA" in m) or has_theta:
                mt = os.path.getmtime(p)
                if mt > best_mtime:
                    best, best_mtime = p, mt
        except Exception:
            continue
    return best  # may be None


def load_theta_epochs(basepath):
    """
    Returns Nx2 float array of [start, stop] seconds for THETA epochs,
    or an empty (0,2) array if not found.
    """
    state_dict = loading.load_SleepState_states(basepath)
    theta_epochs = nel.EpochArray(
    state_dict["THETA"],
    )
    arr = np.asarray(theta_epochs.data, float)
    return arr
def annotate_with_epochs(df_session, epoch_df):
    """
    For each row in df_session, assign:
        - epoch_name: which epoch its center_s falls into
        - environment: epoch_df['environment'] for that epoch

    epoch_df is expected to have columns: 'name', 'startTime', 'stopTime', 'environment'.
    """
    d = df_session.copy()

    if d.empty or epoch_df is None or len(epoch_df) == 0:
        d["epoch_name"] = pd.NA
        d["environment"] = pd.NA
        return d

    # Extract arrays
    names = epoch_df["name"].astype(str).to_numpy()
    starts = pd.to_numeric(epoch_df["startTime"], errors="coerce").to_numpy(float)
    stops  = pd.to_numeric(epoch_df["stopTime"],  errors="coerce").to_numpy(float)
    envs   = epoch_df.get("environment", pd.Series([""] * len(epoch_df))).astype(str).to_numpy()

    # Sort epochs by start time (assumed non-overlapping)
    order = np.argsort(starts)
    starts = starts[order]
    stops  = stops[order]
    names  = names[order]
    envs   = envs[order]

    centers = pd.to_numeric(d["center_s"], errors="coerce").to_numpy(float)

    # For each center: find rightmost start <= center, then check center < stop
    idx = np.searchsorted(starts, centers, side="right") - 1
    valid = (idx >= 0) & (centers < stops[np.clip(idx, 0, len(stops)-1)])

    epoch_name = np.full(centers.shape, None, dtype=object)
    environment = np.full(centers.shape, None, dtype=object)

    good_idx = np.where(valid)[0]
    if good_idx.size > 0:
        epoch_idx = idx[good_idx]
        epoch_name[good_idx] = names[epoch_idx]
        environment[good_idx] = envs[epoch_idx]

    d["epoch_name"] = pd.Series(epoch_name, index=d.index, dtype="object")
    d["environment"] = pd.Series(environment, index=d.index, dtype="object")

    return d


def centers_in_intervals_bool(centers, intervals):
    """
    Vectorized: for each center (1D), test if it falls within ANY [start,stop) row in intervals (Nx2).
    Assumes intervals are unsorted; we sort once.
    """
    centers = np.asarray(centers, float)
    if intervals.size == 0 or centers.size == 0:
        return np.zeros_like(centers, dtype=bool)
    starts = np.asarray(intervals[:,0], float)
    stops  = np.asarray(intervals[:,1], float)
    order = np.argsort(starts)
    starts = starts[order]
    stops  = stops[order]
    # For each center c: find the rightmost start <= c, then check c < corresponding stop
    idx = np.searchsorted(starts, centers, side="right") - 1  # -1 if all starts > c
    ok = (idx >= 0) & (centers < stops[np.clip(idx, 0, len(stops)-1)])
    return ok


def split_wake_into_theta_nontheta(df_session, theta_intervals):
    """
    Takes a single-session DataFrame with columns ['state','center_s', ...].
    Returns a copy where AWAKE rows are split into 'THETA' or 'nonTHETA'
    based on whether center_s ∈ theta_intervals.
    """
    d = df_session.copy()
    awake_mask = d["state"].astype(str).str.upper().eq("AWAKE")
    if awake_mask.any() and theta_intervals.size > 0:
        c = d.loc[awake_mask, "center_s"].to_numpy(float)
        in_theta = centers_in_intervals_bool(c, theta_intervals)
        # assign new state labels
        new_state = d.loc[awake_mask, "state"].copy()
        new_state.values[:] = "nonTHETA"
        new_state.values[in_theta] = "THETA"
        d.loc[awake_mask, "state"] = new_state.values
    else:
        # no θ-epochs found → leave AWAKE as nonTHETA (optional)
        d.loc[awake_mask, "state"] = "nonTHETA"
    return d
# --- DRIVER ---


# ---- data (../csv) ----
pooled, missing = [], []
df_all = read_csv('d2_windows_18sessions.csv')
# ---- the same from the recordings, with analysis_scripts/: ----
# pooled = []
# missing = []
# for basepath in paths:
#     for reg in regions:
#         try:
#             df_sess = load_center_d2_from_session(basepath, reg)
#             if df_sess.empty:
#                 missing.append((basepath, reg))
#                 continue
# 
#             # Split AWAKE → THETA / nonTHETA
#             theta_ep = load_theta_epochs(basepath)    # Nx2
#             df_sess = split_wake_into_theta_nontheta(df_sess, theta_ep)
# 
#             # load epoch table and annotate each row
#             try:
#                 epoch_df = loading.load_epoch(str(basepath))
#             except Exception as e_ep:
#                 print(f"⚠️ Failed to load epoch for {basepath}: {e_ep}")
#                 epoch_df = None
# 
#             df_sess = annotate_with_epochs(df_sess, epoch_df)
# 
#             pooled.append(df_sess)
#         except Exception as e:
#             print(f"⚠️ {reg} @ {basepath}: {e}")
#             missing.append((basepath, reg))
# 
# if not pooled:
#     raise RuntimeError("No usable sessions found.")
# 
# df_all = pd.concat(pooled, ignore_index=True)

# 1) durations from timestamps
df_all = add_duration_from_timestamps(df_all)   # creates 'duration_s'

# 2) equal-width bins (per state label, including THETA/nonTHETA)
N_BINS = 15
BY_WIN = True
df_binned, edges = compute_equal_width_bins(df_all, n_bins=N_BINS, by_window=BY_WIN)

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde
import matplotlib as mpl

import matplotlib as mpl

# Global font: prefer Arial, with safe fallbacks
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
from statsmodels.stats.sandwich_covariance import cov_cluster
import statsmodels.formula.api as smf
from scipy.stats import norm
# -----------------------------
# Helpers
# -----------------------------
def pfmt(p, log10p=None):
    """
    Returns actual scientific notation for extremely small p-values.
    Example: log10p = -120.3 -> '5.01e-121'
    """
    # If we have the log-value, use it to reconstruct the actual magnitude
    if log10p is not None and np.isfinite(log10p):
        exponent = int(np.floor(log10p))
        significand = 10**(log10p - exponent)
        
        if exponent < -3:
            # For Science paper style, use LaTeX-like formatting or standard 'e'
            # return f"{significand:.2f} \times 10^{{{exponent}}}" 
            return f"{significand:.2f}e{exponent}"
        else:
            return f"{10**log10p:.4f}"

    # Fallback if only the float p is provided
    if p is None or not np.isfinite(p):
        return "NA"
    if p == 0: 
        return "0.000" # This shouldn't happen if log10p is passed
    if p < 1e-3:
        return f"{p:.2e}"
    return f"{p:.3f}"


def prepare_for_lmm(df):
    df = df.copy()
    df["animal"] = df["animal"].astype(str)
    df["day"] = df["day"].astype(str)
    df["animal_day"] = df["animal"] + "_day" + df["day"]
    df["state"] = df["state"].astype("category")
    return df

def fit_state_mixedlm(df, states_keep, label="Model", reml=False):
    d = prepare_for_lmm(df)
    d = d[d["state"].isin(states_keep)].copy()
    d = d.dropna(subset=["d2", "state", "animal", "animal_day"])

    if d["animal"].nunique() < 2:
        raise ValueError(f"{label}: insufficient clusters (animals).")

    vc = {"animal_day": "0 + C(animal_day)"}
    m = smf.mixedlm("d2 ~ C(state)", d, groups=d["animal"], vc_formula=vc, re_formula="1")
    
    # Use 'cg' or 'bfgs' if 'lbfgs' fails to converge due to zero variance
    r = m.fit(method="lbfgs", reml=reml, disp=False)

    se_rob = None
    p_rob = None
    
    try:
        # Cluster-robust covariance (Clustering by Animal)
        cov_all = cov_cluster(r, d["animal"])
        k_fe = len(r.fe_params)
        cov_fe = np.asarray(cov_all)[:k_fe, :k_fe]
        fe_names = list(r.fe_params.index)
        se_rob = pd.Series(np.sqrt(np.diag(cov_fe)), index=fe_names)

        p_rob = {}
        for nm in fe_names:
            beta = float(r.fe_params[nm])
            se = float(se_rob[nm])
            if se > 0:
                z_stat = beta / se
                # Use log survival function to avoid underflow at p < 1e-16
                log_p = np.log(2.0) + norm.logsf(abs(z_stat))
                p_rob[nm] = np.exp(log_p) if log_p > -30 else 0.0 # Guard against extreme values
            else:
                p_rob[nm] = np.nan
        p_rob = pd.Series(p_rob)
    except Exception as e:
        print(f"Warning: Robust SE calculation failed: {e}")

    return {"data": d, "model": r, "se_robust": se_rob, "p_robust": p_rob, "label": label}

def print_lmm_key_stats(result, show_intercept=False):
    r = result["model"]
    label = result["label"]
    se_rob = result["se_robust"]
    p_rob = result["p_robust"]

    print(f"\n=== {label}: Statistics (Robustness Focus) ===")
    
    for term in r.fe_params.index:
        if term == "Intercept" and not show_intercept:
            continue

        beta = float(r.fe_params[term])
        
        # Prefer Robust SE and Robust P
        s_err = float(se_rob[term]) if se_rob is not None else float(r.bse_fe[term])
        p_val = float(p_rob[term]) if p_rob is not None else float(r.pvalues[term])
        
        # Calculate 95% CI based on Robust SE
        lo, hi = beta - 1.96 * s_err, beta + 1.96 * s_err

        print(f"{term}:")
        print(f"  β         = {beta:.4f}")
        print(f"  SE (Rob)  = {s_err:.4f}")
        print(f"  p (Robust)= {pfmt(p_val)}")
        print(f"  95% CI    = [{lo:.4f}, {hi:.4f}]")
def plot_density_split(df, by_window=False, title=None, save_path=None, bw_adjust=0.5):
    """
    KDE of D² (x) with each window equally weighted, colored by state.

    If THETA/nonTHETA + 'environment' are present, plot two KDE subplots:
        - left: sleep (NREM, REM)
        - right: wake (THETA, nonTHETA) with environment != 'sleep'

    Also makes a second figure with boxplots of D²:
        - sleep: NREM vs REM
        - wake: THETA vs nonTHETA (env != sleep)

    Otherwise, fall back to a single-panel KDE + single boxplot over all states.
    """
    states = df["state"].astype(str)
    have_theta = states.isin(["THETA", "nonTHETA"]).any()

    # basic palettes
    sleep_palette = {"NREM": "#7b3294", "REM": "#d7b5d8"}
    wake_palette  = {"THETA": "#d95f0e", "nonTHETA": "#fdb863"}
    all_palette   = {**sleep_palette, **wake_palette, "AWAKE": "#d9f0d3"}

    # --- Clean / filter base df ---
    df_plot = df[df["d2"].notna()].copy()
    if df_plot.empty:
        print("No data to plot for density plot.")
        return

    # Decide if we can do the sleep/wake split
    do_split = have_theta and ("environment" in df_plot.columns) and not by_window

    if do_split:
        # ------------------ SPLIT VERSION ------------------
        env = df_plot["environment"].astype(str).str.lower()

        df_sleep = df_plot[df_plot["state"].isin(["NREM", "REM"])].copy()
        df_wake  = df_plot[
            df_plot["state"].isin(["THETA", "nonTHETA"]) &
            (env != "sleep")
        ].copy()
        if df_sleep.empty and df_wake.empty:
            print("No data to plot for sleep/wake split.")
            return

        # ===== KDE FIGURE =====
        fig, axes = plt.subplots(1, 2, figsize=(9.5, 5), sharey=False,sharex=False)
        ax_sleep, ax_wake = axes

        # ---- Sleep panel ----
        if not df_sleep.empty:
            sns.kdeplot(
                data=df_sleep,
                x="d2",
                hue="state",
                hue_order=[s for s in ["REM", "NREM"] if s in df_sleep["state"].unique()],
                palette=sleep_palette,
                weights=None,
                fill=True,
                alpha=0.2,
                common_norm=False,
                bw_adjust=bw_adjust,
                linewidth=3,
                ax=ax_sleep,
            )
            ax_sleep.set_title("Sleep (NREM / REM)")
            ax_sleep.set_xlabel("d2 distance")
            ax_sleep.set_ylabel("Probability density")
            leg = ax_sleep.get_legend()
            if leg is not None:
                leg.remove()
            ax_sleep.spines["top"].set_visible(False)
            ax_sleep.spines["right"].set_visible(False)
            ax_sleep.axvline(0, ls="--", lw=2, color="k")
        else:
            ax_sleep.set_visible(False)

        # ---- Wake panel ----
        if not df_wake.empty:
            sns.kdeplot(
                data=df_wake,
                x="d2",
                hue="state",
                hue_order=[s for s in ["nonTHETA", "THETA"] if s in df_wake["state"].unique()],
                palette=wake_palette,
                weights=None,
                fill=True,
                alpha=0.2,
                common_norm=False,
                bw_adjust=bw_adjust,
                linewidth=3,
                ax=ax_wake,
            )
            ax_wake.set_title("Wake (THETA / nonTHETA, env ≠ sleep)")
          
            ax_wake.set_xlabel("d2 distance")
            ax_wake.set_ylabel("Probability density")
            leg = ax_wake.get_legend()
            if leg is not None:
                leg.remove()
            ax_wake.spines["top"].set_visible(False)
            ax_wake.spines["right"].set_visible(False)
            ax_wake.axvline(0, ls="--", lw=2, color="k")
        else:
            ax_wake.set_visible(False)

        if title:
            fig.suptitle(title, fontweight="bold")
            fig.subplots_adjust(top=0.88)
        ax_sleep.set_xlim(5e-3, 2e-1)
        ax_wake.set_xlim(5e-3, 2e-1)
        plt.tight_layout()

        if save_path is not None:
            Path(save_path).parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(save_path, dpi=300)
            print(f"Saved KDE: {save_path}")

        plt.show()

        # ===== BOX PLOT FIGURE =====
        fig2, axes2 = plt.subplots(1, 2, figsize=(5, 4), sharey=True)
        ax_bx_sleep, ax_bx_wake = axes2

        # Sleep boxplot
        if not df_sleep.empty:
            sleep_order = [s for s in ["NREM", "REM"] if s in df_sleep["state"].unique()]
            sns.boxplot(
                data=df_sleep,
                x="state",
                y="d2",
                order=sleep_order,
                palette=sleep_palette,
                ax=ax_bx_sleep,showfliers=False,width=0.6,linewidth=2
            )
            ax_bx_sleep.set_title("Sleep D² (boxplot)")
            ax_bx_sleep.set_xlabel("State")
            ax_bx_sleep.set_ylabel("d² distance")
            ax_bx_sleep.spines["top"].set_visible(False)
            ax_bx_sleep.spines["right"].set_visible(False)
        else:
            ax_bx_sleep.set_visible(False)

        # Wake boxplot
        if not df_wake.empty:
            wake_order = [s for s in ["THETA", "nonTHETA"] if s in df_wake["state"].unique()]
            sns.boxplot(
                data=df_wake,
                x="state",
                y="d2",
                order=wake_order,
                palette=wake_palette,
                ax=ax_bx_wake,showfliers=False,width=0.6,linewidth=2
            )
            ax_bx_wake.set_title("Wake D² (boxplot, env ≠ sleep)")
            ax_bx_wake.set_xlabel("State")
            ax_bx_wake.set_ylabel("d² distance")
            ax_bx_wake.spines["top"].set_visible(False)
            ax_bx_wake.spines["right"].set_visible(False)
        else:
            ax_bx_wake.set_visible(False)


        plt.tight_layout()

        if save_path is not None:
            box_path = str(Path(save_path).with_suffix("").as_posix()) + "_boxplots.svg"
            plt.savefig(box_path, dpi=300)
            print(f"Saved boxplots: {box_path}")

        plt.show()

    import statsmodels.formula.api as smf

    res_sleep = fit_state_mixedlm(df_sleep, ["NREM", "REM"], label="Sleep (NREM vs REM)", reml=False)
    print(res_sleep["model"].summary())
    print_lmm_key_stats(res_sleep)
    
    # Wake: THETA vs nonTHETA
    res_wake = fit_state_mixedlm(df_wake, ["THETA", "nonTHETA"], label="Wake (THETA vs nonTHETA)", reml=False)
    print(res_wake["model"].summary())
    print_lmm_key_stats(res_wake)
    # ------------------ FALLBACK: single-panel version ------------------
    if have_theta:
        order_states = ["NREM", "REM", "THETA", "nonTHETA"]
    else:
        order_states = ["NREM", "AWAKE", "REM"]

    df_plot = df_plot[df_plot["state"].isin(order_states)].copy()
    if df_plot.empty:
        print("No data to plot for density plot (after state filter).")
        return

df_30s_only = df_binned[df_binned["window"] == "30s"].copy()

if df_30s_only.empty:
    print("No 30s window data found to plot.")
else:
    plot_density_split(
        df_30s_only,
        by_window=False,
        title=None,
        save_path=r"../figure_output\fig1_D_d2_density_sleep_wake.svg",
        bw_adjust=0.5,
    )

FigS1_E_wake_sleep_d2

In [ ]:
# # Global font: prefer Arial, with safe fallbacks
import matplotlib as mpl
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans', 'sans-serif']

# Make mathtext look like Arial
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.it'] = 'Arial:italic'
mpl.rcParams['mathtext.bf'] = 'Arial:bold' 

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
import os, re, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------ I/O helpers ------------------
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(mat_path, squeeze_me=True, struct_as_record=False, simplify_cells=True)
    except TypeError:
        return loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)

def _to_1d_array(x):
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)
    if arr.ndim == 0:
        return np.atleast_1d(arr).astype(float, copy=False)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def _infer_animal_day_from_path(basepath):
    base = os.path.basename(basepath.rstrip("\\/"))
    animal = os.path.basename(os.path.dirname(basepath))
    mday = re.search(r"day(\d+)", base, flags=re.I)
    day = int(mday.group(1)) if mday else None
    return animal, day

# ------------------ extract center_s + d2 ------------------
def _extract_center_d2_by_state_from_mat(mat_path, region=None, animal=None, day=None):
    """
    Returns DataFrame with columns:
      state, center_s, d2, region, animal, day
    """
    mat = _load_mat_any(mat_path)
    results = mat.get("results", None)
    if results is None:
        return pd.DataFrame(columns=["state","center_s","d2","region","animal","day"])

    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    rows = []
    for it in iter_results:
        if hasattr(it, "get"):
            st = _to_state_str(it.get("state", "UNKNOWN"))
            c_raw = it.get("center", [])
            d2_raw = it.get("d2", [])
        elif isinstance(it, dict) or hasattr(it, "__getitem__"):
            st = _to_state_str(it.get("state", it["state"] if "state" in it else "UNKNOWN"))
            c_raw = it.get("center", it["center"] if "center" in it else [])
            d2_raw = it.get("d2", it["d2"] if "d2" in it else [])
        else:
            st, c_raw, d2_raw = "UNKNOWN", [], []

        centers = _to_1d_array(c_raw)
        d2_vals = _to_1d_array(d2_raw)
        ok = np.isfinite(centers) & np.isfinite(d2_vals)
        centers, d2_vals = centers[ok], d2_vals[ok]
        if centers.size == 0:
            continue

        st_up = _to_state_str(st).upper()
        if st_up == "WAKE":
            st_up = "AWAKE"
        if st_up not in ("NREM", "AWAKE", "REM"):
            continue

        rows.append(pd.DataFrame({
            "state": st_up,
            "center_s": centers.astype(float, copy=False),
            "d2": d2_vals.astype(float, copy=False),
            "region": region if region else "",
            "animal": animal if animal else "",
            "day": day if day is not None else ""
        }))

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=["state","center_s","d2","region","animal","day"]
    )

def load_center_d2_from_session(basepath, region):
    """
    Loads newest 30s and newest 60s files (if any) and returns pooled
    center_s + d2 with a 'window' column ∈ {'30s','60s'}.
    """
    region = region.upper()
    search_dir = Path(basepath) / "complex_system"
    if not search_dir.exists():
        return pd.DataFrame(columns=["state","center_s","d2","window","region","animal","day"])

    pats = {
        "30s": f"*_{region}_D2_time_resolved_error_bar.mat",
    }
    animal, day = _infer_animal_day_from_path(basepath)

    out = []
    for win_label, pat in pats.items():
        files = glob.glob(str(search_dir / pat))
        if not files:
            continue
        mat_path = max(files, key=os.path.getmtime)
        df = _extract_center_d2_by_state_from_mat(mat_path, region=region, animal=animal, day=day)
        if df.empty:
            continue
        df["window"] = win_label
        out.append(df)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(
        columns=["state","center_s","d2","window","region","animal","day"]
    )

# --- compute per-row durations from timestamps ---
def add_duration_from_timestamps(df, fill_last="median"):
    """
    For each (animal, day, window, state), sort by center_s and compute row-wise
    duration as the forward difference: duration[i] = center[i+1] - center[i].
    The last row per group gets:
        - fill_last="median": group's median Δt
        - fill_last="drop": NaN (will be dropped later)
        - fill_last=float: a constant seconds value
    Returns a copy with a new 'duration_s' column.
    """
    d = df.copy()
    grp_cols = ["animal","day","window","state"]

    def _durations(g):
        g = g.sort_values("center_s")
        c = g["center_s"].to_numpy(dtype=float)
        dt = np.empty_like(c, dtype=float)
        if c.size == 1:
            dt[:] = np.nan
        else:
            dt[:-1] = np.diff(c)
            # last-row handling
            if fill_last == "median":
                med = np.nanmedian(dt[:-1]) if dt.size > 1 else np.nan
                dt[-1] = med if np.isfinite(med) and med > 0 else np.nan
            elif fill_last == "drop":
                dt[-1] = np.nan
            else:
                # numeric
                try:
                    dt[-1] = float(fill_last)
                except Exception:
                    dt[-1] = np.nan
        # negative/zero dt are invalid
        dt[~np.isfinite(dt) | (dt <= 0)] = np.nan
        g["duration_s"] = dt
        return g

    d = d.groupby(grp_cols, observed=True, group_keys=False).apply(_durations)
    # drop rows with no valid duration
    d = d[np.isfinite(d["duration_s"]) & (d["duration_s"] > 0)]
    return d

def compute_equal_width_bins(d, n_bins=10, by_window=False):
    """
    Equal-width bins on D² within each state (and optionally state×window).
    Adds 'd2_bin' ∈ {1..n_bins} as nullable Int64.
    Returns df_with_bins, edges_map.
    """
    d = d.copy()

    # Make sure d2 is numeric
    d["d2"] = pd.to_numeric(d["d2"], errors="coerce")

    # Pre-create a nullable integer column to avoid dtype thrash
    d["d2_bin"] = pd.Series([pd.NA] * len(d), dtype="Int64")

    edges_map = {}
    groups = (
        d.groupby(["state", "window"], observed=True)
        if by_window else
        d.groupby(["state"], observed=True)
    )

    for key, g in groups:
        idx = g.index
        vals = d.loc[idx, "d2"].to_numpy()

        if vals.size == 0 or not np.isfinite(vals).any():
            continue

        dmin, dmax = float(np.nanmin(vals)), float(np.nanmax(vals))
        if not np.isfinite(dmin) or not np.isfinite(dmax) or dmin == dmax:
            # degenerate case: single bin
            edges = np.array([dmin - 1e-12, dmax + 1e-12], dtype=float)
        else:
            edges = np.linspace(dmin, dmax, n_bins + 1, dtype=float)

        edges_map[key] = edges

        # labels=False -> integer codes [0..n_bins-1], NaN for out-of-bin
        codes = pd.cut(
            d.loc[idx, "d2"],
            bins=edges,
            labels=False,
            include_lowest=True
        )

        # Convert to nullable Int64 with 1-based labels
        # Note: adding 1 keeps NA as NA with pandas nullable integers
        codes = (pd.Series(codes, index=idx, dtype="Int64") + 1).astype("Int64")

        # Single assignment of a clean integer Series
        d.loc[idx, "d2_bin"] = codes

    # Ensure final dtype
    d["d2_bin"] = d["d2_bin"].astype("Int64")
    return d, edges_map





# ------------------ DRIVER ------------------
# Your session folders:
paths = [
    r"W:\data\PPP\PPP20\day4",
    r"W:\data\PPP\PPP20\day5",
    r"W:\data\PPP\PPP20\day7",
    r"W:\data\PPP\PPP20\day8",
    r"W:\data\PPP\PPP20\day9",
    r"W:\data\PPP\PPP21\day4",
    r"W:\data\PPP\PPP21\day5",
    r"W:\data\PPP\PPP21\day7",
    r"W:\data\PPP\PPP21\day8",
    r"W:\data\PPP\PPP8\day8",
    r"W:\data\PPP\PPP8\day14",
    r"W:\data\PPP\PPP4\day11",
    r"W:\data\PPP\PPP7\day7",
    r"W:\data\PPP\PPP7\day10",
    r"W:\data\PPP\PPP7\day12",
    r"W:\data\PPP\PPP7\day14",
    r"W:\data\PPP\PPP23\day7",
    r"W:\data\PPP\PPP23\day8",
]
regions = ["CA1"]  # change if needed
save_root = r"../figure_output"

def _find_state_mat(basepath):
    """
    Heuristics to locate a MAT file holding state_dict/THETA.
    Searches common names in basepath and subfolders.
    """
    candidates = []
    basepath = Path(basepath)
    # common names / locations; extend as needed
    pats = [
        "*state*.mat",
        "*State*.mat",
        "states.mat",
        "state_dict.mat",
        "sleep_states.mat",
        str(basepath / "complex_system" / "*state*.mat"),
        str(basepath / "complex_system" / "state_dict.mat"),
    ]
    for pat in pats:
        candidates.extend(glob.glob(str(basepath / pat) if not pat.startswith(str(basepath)) else pat))
    # fall back: any .mat in basepath
    if not candidates:
        candidates.extend(glob.glob(str(basepath / "*.mat")))
    # pick the newest that actually contains THETA
    best = None
    best_mtime = -1
    for p in candidates:
        try:
            m = _load_mat_any(p)
            # look for THETA under state_dict or top-level
            has_theta = False
            if "state_dict" in m and isinstance(m["state_dict"], dict):
                has_theta = "THETA" in m["state_dict"]
            if ("THETA" in m) or has_theta:
                mt = os.path.getmtime(p)
                if mt > best_mtime:
                    best, best_mtime = p, mt
        except Exception:
            continue
    return best  # may be None


def load_theta_epochs(basepath):
    """
    Returns Nx2 float array of [start, stop] seconds for THETA epochs,
    or an empty (0,2) array if not found.
    """
    state_dict = loading.load_SleepState_states(basepath)
    theta_epochs = nel.EpochArray(
    state_dict["THETA"],
    )
    arr = np.asarray(theta_epochs.data, float)
    return arr
def annotate_with_epochs(df_session, epoch_df):
    """
    For each row in df_session, assign:
        - epoch_name: which epoch its center_s falls into
        - environment: epoch_df['environment'] for that epoch

    epoch_df is expected to have columns: 'name', 'startTime', 'stopTime', 'environment'.
    """
    d = df_session.copy()

    if d.empty or epoch_df is None or len(epoch_df) == 0:
        d["epoch_name"] = pd.NA
        d["environment"] = pd.NA
        return d

    # Extract arrays
    names = epoch_df["name"].astype(str).to_numpy()
    starts = pd.to_numeric(epoch_df["startTime"], errors="coerce").to_numpy(float)
    stops  = pd.to_numeric(epoch_df["stopTime"],  errors="coerce").to_numpy(float)
    envs   = epoch_df.get("environment", pd.Series([""] * len(epoch_df))).astype(str).to_numpy()

    # Sort epochs by start time (assumed non-overlapping)
    order = np.argsort(starts)
    starts = starts[order]
    stops  = stops[order]
    names  = names[order]
    envs   = envs[order]

    centers = pd.to_numeric(d["center_s"], errors="coerce").to_numpy(float)

    # For each center: find rightmost start <= center, then check center < stop
    idx = np.searchsorted(starts, centers, side="right") - 1
    valid = (idx >= 0) & (centers < stops[np.clip(idx, 0, len(stops)-1)])

    epoch_name = np.full(centers.shape, None, dtype=object)
    environment = np.full(centers.shape, None, dtype=object)

    good_idx = np.where(valid)[0]
    if good_idx.size > 0:
        epoch_idx = idx[good_idx]
        epoch_name[good_idx] = names[epoch_idx]
        environment[good_idx] = envs[epoch_idx]

    d["epoch_name"] = pd.Series(epoch_name, index=d.index, dtype="object")
    d["environment"] = pd.Series(environment, index=d.index, dtype="object")

    return d


def centers_in_intervals_bool(centers, intervals):
    """
    Vectorized: for each center (1D), test if it falls within ANY [start,stop) row in intervals (Nx2).
    Assumes intervals are unsorted; we sort once.
    """
    centers = np.asarray(centers, float)
    if intervals.size == 0 or centers.size == 0:
        return np.zeros_like(centers, dtype=bool)
    starts = np.asarray(intervals[:,0], float)
    stops  = np.asarray(intervals[:,1], float)
    order = np.argsort(starts)
    starts = starts[order]
    stops  = stops[order]
    # For each center c: find the rightmost start <= c, then check c < corresponding stop
    idx = np.searchsorted(starts, centers, side="right") - 1  # -1 if all starts > c
    ok = (idx >= 0) & (centers < stops[np.clip(idx, 0, len(stops)-1)])
    return ok


def split_wake_into_theta_nontheta(df_session, theta_intervals):
    """
    Takes a single-session DataFrame with columns ['state','center_s', ...].
    Returns a copy where AWAKE rows are split into 'THETA' or 'nonTHETA'
    based on whether center_s ∈ theta_intervals.
    """
    d = df_session.copy()
    awake_mask = d["state"].astype(str).str.upper().eq("AWAKE")
    if awake_mask.any() and theta_intervals.size > 0:
        c = d.loc[awake_mask, "center_s"].to_numpy(float)
        in_theta = centers_in_intervals_bool(c, theta_intervals)
        # assign new state labels
        new_state = d.loc[awake_mask, "state"].copy()
        new_state.values[:] = "nonTHETA"
        new_state.values[in_theta] = "THETA"
        d.loc[awake_mask, "state"] = new_state.values
    else:
        # no θ-epochs found → leave AWAKE as nonTHETA (optional)
        d.loc[awake_mask, "state"] = "nonTHETA"
    return d
# --- DRIVER ---


# ---- data (../csv) ----
pooled, missing = [], []
df_all = read_csv('d2_windows_18sessions.csv')
# ---- the same from the recordings, with analysis_scripts/: ----
# pooled = []
# missing = []
# for basepath in paths:
#     for reg in regions:
#         try:
#             df_sess = load_center_d2_from_session(basepath, reg)
#             if df_sess.empty:
#                 missing.append((basepath, reg))
#                 continue
# 
#             # Split AWAKE → THETA / nonTHETA
#             theta_ep = load_theta_epochs(basepath)    # Nx2
#             df_sess = split_wake_into_theta_nontheta(df_sess, theta_ep)
# 
#             # load epoch table and annotate each row
#             try:
#                 epoch_df = loading.load_epoch(str(basepath))
#             except Exception as e_ep:
#                 print(f"⚠️ Failed to load epoch for {basepath}: {e_ep}")
#                 epoch_df = None
# 
#             df_sess = annotate_with_epochs(df_sess, epoch_df)
# 
#             pooled.append(df_sess)
#         except Exception as e:
#             print(f"⚠️ {reg} @ {basepath}: {e}")
#             missing.append((basepath, reg))
# 
# if not pooled:
#     raise RuntimeError("No usable sessions found.")
# 
# df_all = pd.concat(pooled, ignore_index=True)



import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
# Map state to group labels
state_to_group = {
    "THETA": "Wake θ",
    "nonTHETA": "Wake non-θ", 
    "NREM": "NREM",
    "REM": "REM"
}

# Create the group column
df_all["group"] = df_all["state"].map(state_to_group)

# (Optional) fix the display order of the state groups
group_order = ["Wake θ", "Wake non-θ", "NREM", "REM"]
if set(group_order).issuperset(set(df_all["group"].unique())):
    df_all["group"] = pd.Categorical(df_all["group"], categories=group_order, ordered=True)

# Combine all CA1 data only
combined_df = df_all
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# (Optional) fix the display order of the state groups
group_order = ["Wake θ", "Wake non-θ", "NREM", "REM"]
if set(group_order).issuperset(set(combined_df["group"].unique())):
    combined_df["group"] = pd.Categorical(combined_df["group"], categories=group_order, ordered=True)

# Plot 2: Simplified Wake vs Sleep grouping
# Create simplified grouping
combined_df_simplified = combined_df.copy()
combined_df_simplified['simplified_group'] = combined_df_simplified['group'].map({
    'Wake θ': 'Wake',
    'Wake non-θ': 'Wake',
    'NREM': 'Sleep',
    'REM': 'Sleep'
})

# Filter out any unmapped values
combined_df_simplified = combined_df_simplified.dropna(subset=['simplified_group'])
combined_df_simplified = combined_df_simplified.dropna(subset=['simplified_group'])

import seaborn as sns
import matplotlib.pyplot as plt


# ---------- FIGURE 2: Simplified Wake vs Sleep ----------
# ---------- FIGURE 2: Simplified Wake vs Sleep (violin with custom colors) ----------
fig2, ax2 = plt.subplots(figsize=(5, 5))
order2   = ['Wake', 'Sleep']
palette2 = {'Wake': '#d95f0e', 'Sleep': '#7b3294'}  # custom colors
sns.violinplot(
    data=combined_df_simplified,
    x="simplified_group", y="d2",
    order=order2, ax=ax2,
    palette=palette2,
    cut=0,              # don't extend beyond data range
    scale='width',      # same width per group
    inner='box',        # show median & IQR inside
    linewidth=1.5
)

ax2.set_ylabel("d2 distance")
ax2.grid(False)
# remove top & right frame lines
for s in ("top", "right"):
    ax2.spines[s].set_visible(False)

# also stop ticks on those sides (optional)
ax2.tick_params(top=False, right=False)

# # n annotations
# counts2 = combined_df_simplified.groupby('simplified_group').size()
# y0, y1 = ax2.get_ylim()
# for i, g in enumerate(order2):
#     if g in counts2.index:
#         ax2.text(i, y0 - 0.05*(y1 - y0), f"n={counts2[g]}",
#                  ha='center', va='top', fontweight='bold')
# Annotate sample sizes below each violin
save_path = r"../figure_output\figS1_B_wake_sleep.svg"
fig2.savefig(save_path, dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import norm, stats
import numpy as np
from scipy.stats import norm

def pfmt_extreme(beta, se, two_sided=True, sig=3):
    """
    Format extremely small p-values safely.

    beta, se: coefficient and standard error
    two_sided: if True -> two-sided p, else one-sided (greater by default)
    sig: significant digits for mantissa in scientific notation

    Returns: string like '2.1e-120' or '< 5e-324' or 'nan'
    """
    beta = float(beta)
    se   = float(se)

    if (not np.isfinite(beta)) or (not np.isfinite(se)) or se <= 0:
        return "nan"

    z = beta / se

    if two_sided:
        # log(p) = log(2) + logsf(|z|)
        logp = np.log(2.0) + norm.logsf(abs(z))
    else:
        # one-sided p for positive effect (greater)
        logp = norm.logsf(z)

    # smallest positive float (subnormal). If smaller, it would print as 0.0
    log_pmin = np.log(np.nextafter(0.0, 1.0))  # about -744.44
    if logp < log_pmin:
        return "< 5e-324"

    p = np.exp(logp)
    # format nicely
    return f"{p:.{sig}e}"

def wake_vs_sleep_session_summary_p(model_df, aggfunc='median'):
    """
    1. Aggregates d2 within each (animal_day, state) to a single value (median/mean).
    2. Ranks those summaries within each animal_day.
    3. Fits a Nested LMM: rank ~ state + (1|animal).
    """
    # 1. Prep and Aggregate
    df = (model_df
          .dropna(subset=['d2','simplified_group','animal','day'])
          .copy())
    df['animal'] = df['animal'].astype(str)
    df['animal_day'] = df['animal'] + '_' + df['day'].astype(str)
    
    # Session-level aggregation (e.g., median d2 per state per day)
    session_df = (df.groupby(['animal', 'animal_day', 'simplified_group'], observed=True)['d2']
                  .agg(aggfunc)
                  .reset_index())
    
    # 2. Rank within each animal_day
    # Since we only have 2 states (Sleep/Wake) per day, 
    # the ranks will be 1 and 2 (or 1.5 if tied).
    session_df['d2_rank'] = session_df.groupby('animal_day')['d2'].rank(method='average')
    
    # 3. Fit Nested LMM
    # With N=15 sessions, we use a simpler intercept for 'animal'
    try:
        model = smf.mixedlm("d2_rank ~ simplified_group", 
                            data=session_df, 
                            groups=session_df["animal"])
        res = model.fit(method='lbfgs', disp=False)
        
        term = 'simplified_group[T.Wake]'
        beta = res.params[term]
        se = res.bse[term]
        
        # Exact P-value calculation
        exact_p_str = pfmt_extreme(beta, se)
        ci = res.conf_int().loc[term].to_numpy()
        
        print(f"=== Session-Level Nested LMM ({aggfunc}) ===")
        print(f"Total Summary Points: {len(session_df)}")
        print(f"Animals: {session_df['animal'].nunique()}, Sessions: {session_df['animal_day'].nunique()}")
        print(f"Δ rank (Wake−Sleep) β = {beta:.4f}")
        print(f"SE = {se:.6f}")
        print(f"p-value = {exact_p_str}")
        
        return np.exp(np.log(2.0) + norm.logsf(abs(beta/se)))

    except Exception as e:
        print(f"Session LMM failed: {e}")
        return np.nan

# Execution
pval_session = wake_vs_sleep_session_summary_p(combined_df_simplified, aggfunc='median')